In [37]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['OPENAI_API_KEY']:
    print("API Key is set.")


API Key is set.


In [7]:
from langchain_openai import ChatOpenAI

In [8]:
llm = ChatOpenAI(model="gpt-5-nano",temperature=0)

RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA
STEP 1: Preparing Document for your Text

In [9]:
from langchain_core.documents import Document

In [18]:
## Your text data
my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of cutting edge AI has filtered into general applications, often without being called AI because once something becomes useful enough and common enough it's not labeled AI anymore."[2][3]

Various subfields of AI research are centered around particular goals and the use of particular tools. The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, perception, and support for robotics.[a] To reach these goals, AI researchers have adapted and integrated a wide range of techniques, including search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[4] Some companies, such as OpenAI, Google DeepMind and Meta,[5] aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.
""" 


docs = [Document(
    page_content=my_text,
    metadata={"source":"ABC","documentID":"Doc1"}
)]
docs

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of c

STEP 2: Splitting the Document into CHUNKS

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50)

chunks = splitter.split_documents(docs) # must use list of Document objects
chunks

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess a

chunks = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
).split_documents(docs)

STEP 3: Creating Embeddings for the Chunks

In [21]:
from langchain_openai import OpenAIEmbeddings

In [23]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

STEP 4: Create and Store Embeddings in Vector Store

In [24]:
from langchain_community.vectorstores import Chroma

In [29]:
vectorstore = Chroma.from_documents(documents=chunks,embedding=embedding_model)

vectorstore = Chroma()
vectorstore.add_documents(chunks, embedding=embedding_model)

Talk to LLM

In [35]:
context = vectorstore.similarity_search("What is ai?",k=3)

invoke() = “send input to the LLM and get a response”

In [36]:
response = llm.invoke(f"What is AI? You can answer using the following context: {context}")
print(response.content)

AI, or artificial intelligence, is the capability of computer systems to perform tasks that are typically associated with human intelligence. These tasks include learning, reasoning, problem-solving, perception, and decision-making. AI is a field of computer science that studies methods and software that let machines perceive their environment, learn, and take actions to achieve defined goals.

Key points:
- Core approaches: neural networks, and methods drawn from statistics, operations research, economics, and other disciplines.
- Interdisciplinary roots: AI also draws on psychology, linguistics, philosophy, neuroscience, and more.
- Goals: Many researchers aim for artificial general intelligence (AGI), which would perform virtually any cognitive task at human-like or better levels.
- Examples of applications: advanced search engines (Google Search), recommendation systems (YouTube, Amazon, Netflix), virtual assistants (Google Assistant, Siri, Alexa), autonomous vehicles (Waymo), gene